In [4]:
import pandas as pd
import numpy as np
import spacy
import re
import sys
sys.path.insert(0,'..')
from source.processing.annotate import annotateSet
from collections import defaultdict
nlp = spacy.load('en_core_web_lg')
pd.options.mode.chained_assignment = None
%matplotlib inline

In [2]:
def url_container(name,url):
    name = name.lower().split()
    url = url.lower().split()
    contains = False
    for string1 in url:
        for string2 in name:
            if string1 == string2:
                contains = True
    return contains

def word_index(text, word, offset):
    text = nlp(text)
    text = [(x.idx, x) for x in text]
    word = word.split()[0]
    counter = 0
    for idx, (pos, value) in enumerate(text):
        if str(value) == word and int(pos - offset) < 10:
            counter = idx
    return counter

def bin_distance(value): 
    if value < 3  : return 1
    elif value < 4  : return 2
    elif value < 5  : return 3
    elif value < 8  : return 4
    elif value < 12 : return 5
    elif value < 16 : return 6
    elif value < 24 : return 7
    elif value < 32 : return 8
    elif value < 64 : return 9

def name_count(text, word):
    try:
        count_total = len(re.findall(word, text))
        text = text.split()
        word = word.split()
        count_first = text.count(word[0])
        count_last = text.count(word[-1])
        return count_first + count_last - count_total
    except:
        return 0

def pronoun_count(text):
    text = nlp(text)
    return len([y for y in text if y.tag_ == 'PRP$'])
    
def process_data(data):
    features = annotateSet(data)
    data['URL'] = data['URL'].map(lambda x : x.replace('http://en.wikipedia.org/wiki/',''))
    data['URL'] = data['URL'].map(lambda x : x.replace('_',' '))
    data['has_a'] = data[['A','URL']].apply(lambda x : url_container(*x), axis=1).astype(int)
    data['has_b'] = data[['B','URL']].apply(lambda x : url_container(*x), axis=1).astype(int)
    data['index_p'] = data[['Text','Pronoun','Pronoun-offset']].apply(lambda x : word_index(*x), axis=1)
    data['index_a'] = data[['Text','A','A-offset']].apply(lambda x : word_index(*x), axis=1)
    data['index_b'] = data[['Text','B','B-offset']].apply(lambda x : word_index(*x), axis=1)
    data['dist_a'] = abs(data['index_p'] - data['index_a']).apply(lambda x : bin_distance(x))
    data['dist_b'] = abs(data['index_p'] - data['index_b']).apply(lambda x : bin_distance(x))
    data = data.drop(['index_a', 'index_b', 'index_p'], axis=1)
    data['name_a'] = data[['Text','A']].apply(lambda x : name_count(*x), axis=1)
    data['name_b'] = data[['Text','B']].apply(lambda x : name_count(*x), axis=1)
    data = data.merge(features, on=['ID'])
    return data

### Train Data

In [3]:
data_1 = pd.read_csv('../../data/download/gap-test.tsv', sep='\t')
data_2 = pd.read_csv('../../data/download/gap-validation.tsv', sep='\t')
data = data_1.append(data_2).reset_index(drop=True)
print('# data:', data.shape)

# data: (2454, 11)


In [4]:
data = process_data(data)
data['fold'] = data.index.map(lambda x : (x % 5) + 1)

/home/ubuntu/anaconda3/lib/python3.6/site-packages/numpy/core/fromnumeric.py:2920: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/home/ubuntu/anaconda3/lib/python3.6/site-packages/numpy/core/_methods.py:85: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [5]:
data.to_csv('../../data/process/data.tsv', index=False, sep='\t')

### Score Data

In [6]:
data = pd.read_csv('../../data/download/gap-development.tsv', sep='\t')
print('# data:', data.shape)

# data: (2000, 11)


In [7]:
data = process_data(data)
data['fold'] = data.index.map(lambda x : (x % 5) + 1)

/home/ubuntu/anaconda3/lib/python3.6/site-packages/numpy/core/fromnumeric.py:2920: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/home/ubuntu/anaconda3/lib/python3.6/site-packages/numpy/core/_methods.py:85: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


In [8]:
data.to_csv('../../data/process/score.tsv', index=False, sep='\t')

### Train Data

In [29]:
def replace_name(text, name, position, tag):
    before_text = text[:position - 20]
    after_text = text[position + 20:]
    replace_text = text[position - 20:position + 20]
    replace_text = replace_text.replace(name, tag)
    text = before_text + replace_text + after_text
    return text

def process_data(data):
    data['Text'] = data[['Text','A','A-offset']].apply(lambda x : replace_name(*x,' [A] '), axis=1)
    data['Text'] = data[['Text','B','B-offset']].apply(lambda x : replace_name(*x,' [B] '), axis=1)
    data['Text'] = data[['Text','Pronoun','Pronoun-offset']].apply(lambda x : replace_name(*x,' [P] '), axis=1)
    return data

In [30]:
data_1 = pd.read_csv('../../data/download/gap-test.tsv', sep='\t')
data_2 = pd.read_csv('../../data/download/gap-validation.tsv', sep='\t')
data = data_1.append(data_2).reset_index(drop=True)
print('# data:', data.shape)

# data: (2454, 11)


In [31]:
data = process_data(data)
data['fold'] = data.index.map(lambda x : (x % 5) + 1)

In [32]:
data.to_csv('../../data/process/data_1.tsv', index=False, sep='\t')

### Score Data

In [38]:
data1 = pd.read_csv('../../data/download/gap-development.tsv', sep='\t')
print('# data:', data.shape)

# data: (2000, 12)


In [34]:
data = process_data(data)
data['fold'] = data.index.map(lambda x : (x % 5) + 1)

In [35]:
data.to_csv('../../data/process/score_1.tsv', index=False, sep='\t')